# 🌿 Exploratory Data Analysis — Deforestation Segmentation

This notebook explores the MultiEarth 2023 satellite imagery dataset for the deforestation segmentation task.

**Contents:**
1. Dataset Overview & Statistics
2. Sample Visualization (RGB composites, individual bands, masks)
3. Spectral Band Analysis
4. Class Balance & Mask Statistics
5. NDVI (Normalized Difference Vegetation Index) Analysis

In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.dataset import (
    load_geotiff, load_mask, select_bands, normalize_image,
    discover_file_pairs, create_synthetic_dataset, IMG_SIZE, NUM_BANDS
)
from src.utils import get_rgb_from_multiband

# Plot settings
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print('✓ All imports successful')

## 1. Dataset Overview

Load the dataset and compute basic statistics.

In [ ]:
# Configure data directory
DATA_DIR = os.path.join('..', 'data', 'multiearth')
USE_SYNTHETIC = not os.path.exists(DATA_DIR) or len(os.listdir(DATA_DIR)) == 0

if USE_SYNTHETIC:
    print('⚠️ Real data not found. Generating synthetic data for EDA demo...')
    from src.download_data import generate_synthetic_geotiffs
    os.makedirs(DATA_DIR, exist_ok=True)
    generate_synthetic_geotiffs(DATA_DIR, num_samples=50, img_size=256)

# Discover file pairs
file_pairs = discover_file_pairs(DATA_DIR)
print(f'\nDataset size: {len(file_pairs)} image-mask pairs')
print(f'Data directory: {os.path.abspath(DATA_DIR)}')

# Show first few files
for img, mask in file_pairs[:3]:
    print(f'  Image: {os.path.basename(img)}')
    print(f'  Mask:  {os.path.basename(mask)}')

## 2. Sample Visualization

In [ ]:
# Load and display sample images
num_samples = min(4, len(file_pairs))
fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
if num_samples == 1:
    axes = axes[np.newaxis, :]

mask_cmap = mcolors.ListedColormap(['#2d6a4f', '#e63946'])

for i in range(num_samples):
    img_path, mask_path = file_pairs[i]
    
    # Load
    image = load_geotiff(img_path)
    mask = load_mask(mask_path)
    
    # Select bands and normalize
    image = select_bands(image)
    image = normalize_image(image)
    rgb = get_rgb_from_multiband(image)
    
    # Plot RGB
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(f'RGB Composite — Sample {i+1}', fontweight='bold')
    axes[i, 0].axis('off')
    
    # Plot mask
    axes[i, 1].imshow(mask.squeeze(), cmap=mask_cmap, vmin=0, vmax=1)
    axes[i, 1].set_title(f'Deforestation Mask', fontweight='bold')
    axes[i, 1].axis('off')
    
    # Plot overlay
    axes[i, 2].imshow(rgb)
    overlay = np.zeros((*mask.squeeze().shape, 4))
    overlay[mask.squeeze() > 0.5] = [1, 0, 0, 0.4]
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'Overlay (Red = Deforested)', fontweight='bold')
    axes[i, 2].axis('off')

plt.suptitle('Sample Satellite Images with Deforestation Masks', 
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Spectral Band Analysis

Analyze the distribution of pixel values in each spectral band.

In [ ]:
# Collect band statistics
band_names = ['Blue (B2)', 'Green (B3)', 'Red (B4)', 'NIR (B8)']
band_colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']

all_band_values = {name: [] for name in band_names}

sample_count = min(20, len(file_pairs))
for img_path, _ in file_pairs[:sample_count]:
    image = load_geotiff(img_path)
    image = select_bands(image)
    for b, name in enumerate(band_names[:image.shape[-1]]):
        all_band_values[name].extend(image[..., b].flatten().tolist()[:1000])

# Plot distributions
fig, axes = plt.subplots(1, len(band_names), figsize=(18, 5))
for i, (name, color) in enumerate(zip(band_names, band_colors)):
    if all_band_values[name]:
        axes[i].hist(all_band_values[name], bins=50, color=color, alpha=0.7, edgecolor='white')
        axes[i].set_title(name, fontweight='bold', fontsize=13)
        axes[i].set_xlabel('Pixel Value')
        axes[i].set_ylabel('Frequency')
        axes[i].axvline(np.mean(all_band_values[name]), color='black', 
                        linestyle='--', label=f'Mean: {np.mean(all_band_values[name]):.4f}')
        axes[i].legend()

plt.suptitle('Spectral Band Value Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Class Balance Analysis

Check the ratio of forested vs deforested pixels across the dataset.

In [ ]:
# Compute per-image deforestation percentages
deforestation_pcts = []

for _, mask_path in file_pairs:
    mask = load_mask(mask_path)
    pct = mask.mean() * 100
    deforestation_pcts.append(pct)

deforestation_pcts = np.array(deforestation_pcts)

# Summary statistics
print('='*50)
print('  CLASS BALANCE STATISTICS')
print('='*50)
print(f'  Total images:          {len(deforestation_pcts)}')
print(f'  Mean deforestation:    {deforestation_pcts.mean():.2f}%')
print(f'  Median deforestation:  {np.median(deforestation_pcts):.2f}%')
print(f'  Std deforestation:     {deforestation_pcts.std():.2f}%')
print(f'  Min deforestation:     {deforestation_pcts.min():.2f}%')
print(f'  Max deforestation:     {deforestation_pcts.max():.2f}%')
print(f'  Images with >50% def:  {(deforestation_pcts > 50).sum()}')
print(f'  Images with 0% def:    {(deforestation_pcts == 0).sum()}')
print('='*50)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(deforestation_pcts, bins=20, color='#e63946', alpha=0.7, edgecolor='white')
ax1.axvline(deforestation_pcts.mean(), color='black', linestyle='--', 
            label=f'Mean: {deforestation_pcts.mean():.1f}%')
ax1.set_xlabel('Deforestation Percentage (%)', fontsize=12)
ax1.set_ylabel('Number of Images', fontsize=12)
ax1.set_title('Distribution of Deforestation Coverage', fontweight='bold', fontsize=13)
ax1.legend()

# Pie chart
avg_forest = 100 - deforestation_pcts.mean()
avg_deforested = deforestation_pcts.mean()
ax2.pie([avg_forest, avg_deforested], 
        labels=['Forest', 'Deforested'],
        colors=['#2d6a4f', '#e63946'],
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 12})
ax2.set_title('Average Class Distribution', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

## 5. NDVI Analysis

The **Normalized Difference Vegetation Index (NDVI)** is a key indicator for vegetation health:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

- NDVI > 0.3: Dense vegetation (healthy forest)
- NDVI 0.1–0.3: Sparse vegetation
- NDVI < 0.1: Bare soil / deforested

In [ ]:
# Compute NDVI for sample images
num_ndvi = min(4, len(file_pairs))
fig, axes = plt.subplots(num_ndvi, 3, figsize=(15, 5 * num_ndvi))
if num_ndvi == 1:
    axes = axes[np.newaxis, :]

for i in range(num_ndvi):
    img_path, mask_path = file_pairs[i]
    image = load_geotiff(img_path)
    image = select_bands(image)
    mask = load_mask(mask_path)
    
    if image.shape[-1] >= 4:
        red = image[..., 2].astype(np.float64)
        nir = image[..., 3].astype(np.float64)
        
        # Compute NDVI
        ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red + 1e-8), 0)
    else:
        # Fallback if no NIR band
        ndvi = np.zeros_like(image[..., 0])
    
    # RGB
    rgb = get_rgb_from_multiband(normalize_image(image))
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(f'RGB — Sample {i+1}', fontweight='bold')
    axes[i, 0].axis('off')
    
    # NDVI
    im = axes[i, 1].imshow(ndvi, cmap='RdYlGn', vmin=-0.5, vmax=1.0)
    axes[i, 1].set_title('NDVI', fontweight='bold')
    axes[i, 1].axis('off')
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)
    
    # Mask
    axes[i, 2].imshow(mask.squeeze(), cmap=mask_cmap, vmin=0, vmax=1)
    axes[i, 2].set_title('Deforestation Mask', fontweight='bold')
    axes[i, 2].axis('off')

plt.suptitle('NDVI Analysis — Vegetation Health vs Deforestation', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Data Pipeline Test

Verify the `tf.data` pipeline works correctly.

In [ ]:
import tensorflow as tf
from src.dataset import create_synthetic_dataset

# Create a test dataset
test_ds = create_synthetic_dataset(num_samples=16, batch_size=4, is_train=True)

# Verify shapes and values
for batch_imgs, batch_masks in test_ds.take(1):
    print(f'Batch images shape: {batch_imgs.shape}')
    print(f'Batch masks shape:  {batch_masks.shape}')
    print(f'Image dtype: {batch_imgs.dtype}')
    print(f'Mask dtype:  {batch_masks.dtype}')
    print(f'Image range: [{batch_imgs.numpy().min():.3f}, {batch_imgs.numpy().max():.3f}]')
    print(f'Mask values: {np.unique(batch_masks.numpy())}')
    
    # Plot batch
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i in range(4):
        rgb = get_rgb_from_multiband(batch_imgs[i].numpy())
        axes[0, i].imshow(rgb)
        axes[0, i].set_title(f'Image {i+1}', fontweight='bold')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(batch_masks[i].numpy().squeeze(), cmap=mask_cmap, vmin=0, vmax=1)
        axes[1, i].set_title(f'Mask {i+1}', fontweight='bold')
        axes[1, i].axis('off')
    
    plt.suptitle('tf.data Pipeline — Sample Batch', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\n✓ tf.data pipeline test passed!')

## 7. Summary

### Key Findings

1. **Class Imbalance:** Deforested regions typically cover a small portion of each image → justifies using Dice + Focal loss
2. **Spectral Signatures:** Deforested areas show higher Red reflectance and lower NIR compared to forest → NDVI is a strong discriminator
3. **Spatial Patterns:** Deforestation patches vary from small clearings to large blocks → multi-scale features from EfficientNet encoder are important

### Recommendations for Training

- Use **strong augmentation** (flip, rotate, noise) to improve generalization
- **4-band input** (RGB + NIR) provides more signal than RGB alone
- **Dice + Focal loss** combination essential for handling class imbalance
- Consider **class-weighted sampling** if imbalance is severe